In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
from scipy.stats import zscore

plt.rcParams["font.family"] = "Malgun Gothic"   # Windows
plt.rcParams["axes.unicode_minus"] = False

In [2]:
analysis = pd.read_csv('../../data/busan_youth_v4.csv', index_col='Unnamed: 0')
analysis.head(2)

,연도,행정구역,연앙인구수(천명),재정자립도(%),통계방식변경,상용월평균급여(원),문화예술활동(건),대학수(천명당),문화기반시설수(십만명당),기업밀도,지역내총생산(십억원),청년_19_29_증감률(%),청년_30_39_증감률(%)
0,2016,강원특별자치도,1536.2750,22.40,0.0,3023906.0,1444.0,0.012368,13.734520,351.305988,44677.444,1.149646,-2.070627
1,2016,경기도,12509.4835,61.61,0.0,3328379.0,3925.0,0.006235,4.028943,2230.077238,433443.961,1.820153,-0.773010


In [3]:
analysis.columns

Index(['연도', '행정구역', '연앙인구수(천명)', '재정자립도(%)', '통계방식변경', '상용월평균급여(원)',
       '문화예술활동(건)', '대학수(천명당)', '문화기반시설수(십만명당)', '기업밀도', '지역내총생산(십억원)',
       '청년_19_29_증감률(%)', '청년_30_39_증감률(%)'],
      dtype='object')

In [4]:
# 2차 변수 분류
# 종속변수
y_cols = analysis[['청년_19_29_증감률(%)', '청년_30_39_증감률(%)']]

# 독립변수
x_cols = analysis[['상용월평균급여(원)', '문화예술활동(건)', 
                '대학수(천명당)', '문화기반시설수(십만명당)', '기업밀도']]

# 모델링용 독립변수
x_cols_control = analysis[['상용월평균급여(원)', '문화예술활동(건)', 
                '대학수(천명당)', '문화기반시설수(십만명당)', '기업밀도', '통계방식변경']]


# 통제변수
meta_cols = analysis[['연앙인구수(천명)', '재정자립도(%)', '지역내총생산(십억원)']]

# Group by
group_cols = analysis[['연도', '행정구역']]

In [5]:
def add_yearly_zscore(df, col, group="연도"):
    mean = df.groupby(group)[col].transform("mean")
    std = df.groupby(group)[col].transform("std")
    return (df[col] - mean) / std


In [6]:
analysis_z = analysis.copy()

analysis_z['청년_19_29_증감률(%)_z'] = add_yearly_zscore(
    analysis_z, '청년_19_29_증감률(%)'
)

analysis_z['청년_30_39_증감률(%)_z'] = add_yearly_zscore(
    analysis_z, '청년_30_39_증감률(%)'
)


---

z-score 만 csv 파일로 저장

In [7]:
x_vars = [
    '상용월평균급여(원)',
    '문화예술활동(건)',
    '대학수(천명당)',
    '문화기반시설수(십만명당)',
    '기업밀도'
]

for col in x_vars:
    analysis_z[f'{col}_z'] = add_yearly_zscore(analysis_z, col)


In [8]:
meta_vars = [
    '연앙인구수(천명)',
    '재정자립도(%)',
    '지역내총생산(십억원)'
]

for col in meta_vars:
    analysis_z[f'{col}_z'] = add_yearly_zscore(analysis_z, col)


In [9]:
analysis_z

,연도,행정구역,연앙인구수(천명),재정자립도(%),통계방식변경,상용월평균급여(원),문화예술활동(건),대학수(천명당),문화기반시설수(십만명당),기업밀도,...,청년_19_29_증감률(%)_z,청년_30_39_증감률(%)_z,상용월평균급여(원)_z,문화예술활동(건)_z,대학수(천명당)_z,문화기반시설수(십만명당)_z,기업밀도_z,연앙인구수(천명)_z,재정자립도(%)_z,지역내총생산(십억원)_z
0,2016,강원특별자치도,1536.2750,22.40,0.0,3023906.0,1444.0,0.012368,13.734520,351.305988,...,-0.119064,-0.284835,-0.600234,-0.205085,0.854703,1.549660,-0.387657,-0.455941,-1.188626,-0.520361
1,2016,경기도,12509.4835,61.61,0.0,3328379.0,3925.0,0.006235,4.028943,2230.077238,...,0.014971,-0.045046,0.166934,0.680446,-0.923381,-0.598474,3.040583,2.946692,1.069933,2.677738
2,2016,경상남도,3348.2515,36.61,0.0,3257296.0,1689.0,0.007168,5.764203,652.882347,...,-0.359782,-0.291791,-0.012171,-0.117638,-0.652955,-0.214410,0.162636,0.105927,-0.370107,0.045773
3,2016,경상북도,2683.4775,26.10,0.0,3326347.0,1035.0,0.014533,7.266690,549.557763,...,-0.370028,-0.304519,0.161814,-0.351067,1.482685,0.118136,-0.025902,-0.100210,-0.975500,0.035230
4,2016,광주광역시,1461.5310,46.68,0.0,2994355.0,1284.0,0.012316,3.900020,289.682121,...,-0.426919,-0.547518,-0.674692,-0.262193,0.839705,-0.627009,-0.500104,-0.479118,0.209941,-0.563549
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148,2024,전라남도,1792.3250,24.40,1.0,4034870.0,1438.0,0.010601,13.390429,504.474684,...,-0.518022,0.005958,-0.162273,-0.358235,0.406573,1.182580,-0.306634,-0.359256,-1.137119,-0.260644
149,2024,전북특별자치도,1741.6360,23.51,1.0,3736595.0,1839.0,0.011483,10.794448,537.972610,...,-0.972103,-0.271329,-0.882036,-0.251928,0.669217,0.598846,-0.265283,-0.374308,-1.201416,-0.468302
150,2024,제주특별자치도,669.0510,34.01,1.0,3501134.0,1053.0,0.005979,18.832645,202.652238,...,-0.976313,-0.769400,-1.450222,-0.460300,-0.968725,2.406320,-0.679208,-0.692800,-0.442855,-0.690959
151,2024,충청남도,2126.6115,32.35,1.0,4680157.0,1832.0,0.011286,9.357610,545.099532,...,1.911852,0.311186,1.394857,-0.253783,0.610334,0.275757,-0.256486,-0.259994,-0.562780,0.000253


In [10]:
[c for c in analysis_z.columns if c.endswith('_z')]

['청년_19_29_증감률(%)_z',
 '청년_30_39_증감률(%)_z',
 '상용월평균급여(원)_z',
 '문화예술활동(건)_z',
 '대학수(천명당)_z',
 '문화기반시설수(십만명당)_z',
 '기업밀도_z',
 '연앙인구수(천명)_z',
 '재정자립도(%)_z',
 '지역내총생산(십억원)_z']

In [11]:
analysis_z.columns

Index(['연도', '행정구역', '연앙인구수(천명)', '재정자립도(%)', '통계방식변경', '상용월평균급여(원)',
       '문화예술활동(건)', '대학수(천명당)', '문화기반시설수(십만명당)', '기업밀도', '지역내총생산(십억원)',
       '청년_19_29_증감률(%)', '청년_30_39_증감률(%)', '청년_19_29_증감률(%)_z',
       '청년_30_39_증감률(%)_z', '상용월평균급여(원)_z', '문화예술활동(건)_z', '대학수(천명당)_z',
       '문화기반시설수(십만명당)_z', '기업밀도_z', '연앙인구수(천명)_z', '재정자립도(%)_z',
       '지역내총생산(십억원)_z'],
      dtype='object')

In [12]:
# z-score 컬럼만 추출
z_cols = [c for c in analysis_z.columns if c.endswith('_z')]

# 연도 + 행정구역 + z-score 컬럼만 선택
analysis_z_only = analysis_z[['연도', '행정구역'] + z_cols]

analysis_z_only.head()


,연도,행정구역,청년_19_29_증감률(%)_z,청년_30_39_증감률(%)_z,상용월평균급여(원)_z,문화예술활동(건)_z,대학수(천명당)_z,문화기반시설수(십만명당)_z,기업밀도_z,연앙인구수(천명)_z,재정자립도(%)_z,지역내총생산(십억원)_z
0,2016,강원특별자치도,-0.119064,-0.284835,-0.600234,-0.205085,0.854703,1.549660,-0.387657,-0.455941,-1.188626,-0.520361
1,2016,경기도,0.014971,-0.045046,0.166934,0.680446,-0.923381,-0.598474,3.040583,2.946692,1.069933,2.677738
2,2016,경상남도,-0.359782,-0.291791,-0.012171,-0.117638,-0.652955,-0.214410,0.162636,0.105927,-0.370107,0.045773
3,2016,경상북도,-0.370028,-0.304519,0.161814,-0.351067,1.482685,0.118136,-0.025902,-0.100210,-0.975500,0.035230
4,2016,광주광역시,-0.426919,-0.547518,-0.674692,-0.262193,0.839705,-0.627009,-0.500104,-0.479118,0.209941,-0.563549


In [13]:
analysis_z_only.to_csv(
    '../../data/busan_youth_zscore_only.csv',
    encoding='utf-8-sig',
    index=False
)

In [15]:
busan_z_mean = (
    analysis_z[analysis_z['행정구역'] == '부산광역시']
    [['청년_19_29_증감률(%)_z', '청년_30_39_증감률(%)_z']
    + [f'{c}_z' for c in x_vars]
    + [f'{c}_z' for c in meta_vars]]
    .mean()
).sort_values()   # 보기 좋게 정렬(선택)


busan_z_mean


문화기반시설수(십만명당)_z     -0.836676
대학수(천명당)_z          -0.675158
청년_19_29_증감률(%)_z   -0.613619
상용월평균급여(원)_z        -0.560950
청년_30_39_증감률(%)_z   -0.320986
지역내총생산(십억원)_z       -0.151334
문화예술활동(건)_z          0.105148
연앙인구수(천명)_z          0.107177
기업밀도_z               0.180295
재정자립도(%)_z           0.438308
dtype: float64